
# Running AI Functions to do Job Error Classification

We should have gotten the different classification "topics" based on our topic clustering notebook. Those results should be stored in `job_error_classes_table`.

Now, we're going to use AI Functions to classify our detailed error messages. This can be run on a regular basis (daily)

In [0]:
from pyspark.sql import functions as F

In [0]:
# Enable automatic schema evolution
# spark.sql("SET spark.databricks.delta.schema.autoMerge.enabled = true") 

In [0]:
%python
dbutils.widgets.text("target_catalog", "field_demos", "Target Catalog")
dbutils.widgets.text("target_schema", "tauherng", "Target Schema")
dbutils.widgets.text("target_table", "job_errors", "Target Table")
dbutils.widgets.text(
    "job_error_classes_table",
    "job_error_cluster_descriptions",
    "Error Classification Table",
)

In [0]:
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")
job_error_classes_table = dbutils.widgets.get("job_error_classes_table")

In [0]:
%sql
WITH all_descriptions AS (
  SELECT DISTINCT
    (CLUSTER_DESCRIPTION)
  FROM
    IDENTIFIER(CONCAT(:target_catalog, ".", :target_schema, ".", :job_error_classes_table))
  ORDER BY
    CLUSTER_DESCRIPTION
),
deduped AS (
  SELECT
    parse_json(
      replace(
        replace(
          ai_query(
            "databricks-claude-sonnet-4",
            "Help to deduplicate these and return a JSON-formatted string as the deduplicated results. No need for any explanation. I just want the JSON: "
            || to_json(collect_set(cluster_description))
          ),
          "\`\`\`json",
          ""
        ),
        "\`\`\`",
        ""
      )
    ) AS deduplicated_descriptions
  from
    all_descriptions
)
SELECT
  EXPLODE(CAST(deduplicated_descriptions AS ARRAY<STRING>))
from
  deduped;
-- Note that we have some duplicate classes, but for now we will not clean them up. Can clean them up in the final stage (manual) if required.

In [0]:
classes = [r.col for r in _sqldf.select("col").distinct().collect()]

In [0]:
classes

In [0]:
query = f"""
SELECT run_id, error, error_trace, AI_CLASSIFY(CONCAT(regexp_replace(coalesce(error, ""), '[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}', '<uuid>'), COALESCE(error_trace, "")), ARRAY({", ".join([f"'{c}'" for c in classes])})) AS error_classification
FROM
  {target_catalog}.{target_schema}.{target_table}_holding
"""

df = spark.sql(query).withColumn("update_time", F.current_timestamp())

In [0]:
# df.display()

In [0]:
df.createOrReplaceTempView("updates")

merge_sql = f"""
MERGE INTO {dbutils.widgets.get('target_catalog')}.{dbutils.widgets.get('target_schema')}.{dbutils.widgets.get('target_table')} AS target
USING updates
ON updates.run_id = target.run_id
WHEN MATCHED THEN
  UPDATE SET
    target.error = updates.error,
    target.error_trace = updates.error_trace,
    target.update_time = updates.update_time,
    target.error_classification = updates.error_classification
WHEN NOT MATCHED THEN
  INSERT *
"""

spark.sql(merge_sql)

In [0]:
%sql
select * from IDENTIFIER(CONCAT(:target_catalog, ".", :target_schema, ".", :target_table))